<a href="https://colab.research.google.com/github/baohuy1303/scraper_v4/blob/main/scraper_v3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install camoufox rich aiohttp
# !pip install playwright
# !playwright install chromium
# !playwright install-deps chromium
!camoufox fetch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 MB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.4/213.4 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 5.6 MB/s eta 0:00:00
173.7 MiB [] 0% 0.0s173.7 MiB [] 0% 7.8s173.7 MiB [] 0% 3.6s173.7 MiB [] 1% 3.1s173.7 MiB [] 2% 3.1s173.7 MiB [] 2% 3.3s173.7 MiB [] 3% 3.0s173.7 MiB [] 4% 2.7s173.7 MiB [] 4% 2.5s173.7 MiB [] 5% 2.6s173.7 MiB [] 6% 2.6s173.7 MiB [] 7% 2.4s173.7 MiB [] 8% 2.3s173.7 MiB [] 9% 2.3s173.7 MiB [] 10% 2.2s173.7 MiB [] 11% 2.1s173.7 MiB [] 12% 1.9s173.7 MiB [] 13% 2.0s173.7 MiB [] 14% 1.9s173.7 MiB [] 15% 1.8s173.7 MiB [] 16% 1.7s173.7 MiB [] 18% 1.7s173.7 MiB [] 19% 1.6s173.7 MiB [] 20% 1.6s173.7 MiB [] 21% 1.5s173.7 MiB [] 22% 1.4s173.7 MiB [] 24% 1.4s173.7 MiB [] 25% 1.4s173.7 MiB [] 26% 1.4s173.7 MiB [] 27% 1.4s173.7 MiB [] 28% 1.3s173.7 MiB [] 30% 1.3s173.7 MiB [] 32% 1.2s173.7

In [ ]:
from pathlib import Path

# OUTPUT_DIR = Path("/content/drive/MyDrive/Magic_Hour_Scraper/scraped_data")
OUTPUT_DIR = Path("scraped_data")
NUM_WORKERS = 5
BATCH_SIZE = 50
MIN_FILE_SIZE = 1500

PAGE_LOAD_WAIT = 2000

def get_scroll_config(total_media: int) -> tuple[float, int]:
    if total_media < 10:
        return 0.10, 50
    elif total_media < 50:
        return 0.05, 75
    else:
        return 0.02, 150


In [ ]:
# @title
import asyncio
import json
import sys
import re
import hashlib
from typing import Any, Awaitable, Callable, Literal
from datetime import datetime
from dataclasses import dataclass
from urllib.parse import urlparse

import aiofiles
from playwright.async_api import Page, Response
from camoufox.async_api import AsyncCamoufox
from rich.console import Console

console = Console()

# URL patterns to skip (trackers, analytics, pixels, etc.)
SKIP_URL_PATTERNS = [
    "favicon.ico",
    "google-analytics",
    "googletagmanager",
    "doubleclick.net",
    "facebook.com/tr",
    "analytics.js",
    "gtag/js",
    "pixel",
    "tracker",
    "1x1",
    "beacon",
    "telemetry",
    "metrics",
]


@dataclass
class DownloadTask:
    url: str
    filepath: Path
    body: bytes
    resource_type: str
    filename: str


async def download_worker(
    queue: asyncio.Queue,
    captured_files: dict[str, list[dict[str, Any]]],
    session_dir: Path,
):
    write_buffer: list[tuple[DownloadTask, bytes]] = []

    while True:
        task: DownloadTask | None = await queue.get()
        if task is None:
            queue.task_done()
            if write_buffer:
                await _flush_buffer(write_buffer, captured_files)
            break

        try:
            write_buffer.append((task, task.body))
            queue.task_done()

            if len(write_buffer) >= BATCH_SIZE:
                await _flush_buffer(write_buffer, captured_files)
                write_buffer = []
        except Exception as e:
            queue.task_done()


async def _flush_buffer(
    buffer: list[tuple[DownloadTask, bytes]],
    captured_files: dict[str, list[dict[str, Any]]],
):
    """Write all buffered files to disk concurrently."""
    if not buffer:
        return

    await asyncio.gather(
        *[_write_file(task, body, captured_files) for task, body in buffer]
    )

    console.print(
        f"[dim green]Flushed buffer:[/dim green] {len(buffer)} files written to disk"
    )


async def _write_file(
    task: DownloadTask,
    body: bytes,
    captured_files: dict[str, list[dict[str, Any]]],
):
    """Write a single file to disk."""
    try:
        async with aiofiles.open(task.filepath, "wb") as f:
            await f.write(body)

        file_info = {
            "url": task.url,
            "filename": task.filename,
            "saved_path": str(task.filepath),
            "size_bytes": len(body),
        }
        captured_files["images" if task.resource_type == "image" else "videos"].append(
            file_info
        )
    except Exception as e:
        console.print(f"[dim red]Failed to write {task.filename}:[/dim red] {e}")


def get_extension_from_content_type(
    content_type: str | None, resource_type: str
) -> str:
    if not content_type:
        return "png" if resource_type == "image" else "mp4"

    mime_to_ext = {
        "image/jpeg": "jpg",
        "image/jpg": "jpg",
        "image/png": "png",
        "image/gif": "gif",
        "image/webp": "webp",
        "image/svg+xml": "svg",
        "image/avif": "avif",
        "image/bmp": "bmp",
        "image/x-icon": "ico",
        "video/mp4": "mp4",
        "video/webm": "webm",
        "video/ogg": "ogv",
        "video/quicktime": "mov",
        "video/x-msvideo": "avi",
        "video/x-matroska": "mkv",
    }
    main_type = content_type.split(";")[0].strip().lower()
    return mime_to_ext.get(main_type, "png" if resource_type == "image" else "mp4")


async def on_response(
    response: Response,
    download_queue: asyncio.Queue,
    seen_urls: set[str],
    session_dir: Path,
):
    try:
        if not response.ok or response.url in seen_urls:
            return

        resource_type = response.request.resource_type
        if resource_type not in ("image", "media"):
            return

        url = response.url

        # Skip known trackers, analytics, and pixel patterns
        url_lower = url.lower()
        if any(pattern in url_lower for pattern in SKIP_URL_PATTERNS):
            return

        seen_urls.add(url)

        content_length = response.headers.get("content-length")
        if content_length and int(content_length) < MIN_FILE_SIZE:
            return

        content_type = response.headers.get("content-type")
        ext = get_extension_from_content_type(content_type, resource_type)

        url_filename = url.split("/")[-1].split("?")[0]
        if url_filename and "." in url_filename:
            base_filename = url_filename.rsplit(".", 1)[0]
        else:
            base_filename = url_filename or f"{resource_type}_{hash(url)}"

        url_hash = hashlib.md5(url.encode()).hexdigest()[:8]
        filename = f"{base_filename}_{url_hash}.{ext}"

        output_dir = session_dir / ("images" if resource_type == "image" else "videos")
        filepath = output_dir / filename

        body = await response.body()

        if len(body) < MIN_FILE_SIZE:
            return

        download_queue.put_nowait(
            DownloadTask(url, filepath, body, resource_type, filename)
        )
    except Exception as e:
        console.print(f"[dim red]Failed to capture {response.url}:[/dim red] {e}")


async def extract_text(page: Page) -> dict[str, Any]:
    console.print("[cyan]Extracting text content...[/cyan]")
    all_texts = await page.locator("body").all_inner_texts()
    cleaned_texts = [text.strip() for text in all_texts if text.strip()]
    return {"name": "extract_text", "data": cleaned_texts, "count": len(cleaned_texts)}


async def extract_metadata(page: Page) -> dict[str, Any]:
    console.print("[cyan]Extracting metadata...[/cyan]")

    # Single JS call to get all metadata at once (much faster than multiple locator queries)
    metadata = await page.evaluate(
        """
        () => {
            const title = document.title;
            const description = document.querySelector('meta[name="description"]')?.content || null;
            const ogImages = Array.from(document.querySelectorAll('meta[property="og:image"]'))
                .map(el => el.content)
                .filter(Boolean);

            return {
                title,
                description,
                og_images: ogImages,
                og_image: ogImages[0] || null
            };
        }
    """
    )

    return {
        "name": "extract_metadata",
        "data": metadata,
    }


async def adaptive_scroll(page: Page) -> dict[str, int]:
    img_count = await page.locator("img").count()
    video_count = await page.locator("video").count()
    total_media = img_count + video_count

    console.print(
        f"[dim]Found {img_count} images and {video_count} videos on initial page load[/dim]"
    )

    scroll_percent, delay = get_scroll_config(total_media)

    console.print(
        f"[dim]Using {scroll_percent*100}% scroll step with {delay}ms delay[/dim]"
    )

    await page.evaluate(
        f"""
        async () => {{
            const totalHeight = document.scrollingElement.scrollHeight;
            const viewportHeight = window.innerHeight;
            const scrollableHeight = totalHeight - viewportHeight;
            const scrollStep = Math.min(Math.max(scrollableHeight * {scroll_percent}, 100), 500);
            const delay = {delay};

            while (document.scrollingElement.scrollTop + viewportHeight < totalHeight) {{
                document.scrollingElement.scrollBy(0, scrollStep);
                await new Promise(resolve => setTimeout(resolve, delay));
            }}
        }}
        """
    )

    # return {
    #     "initial_images": img_count,
    #     "initial_videos": video_count,
    #     "total_media": total_media,
    #     "scroll_percent": scroll_percent,
    #     "delay": delay,
    # }


ProcessorType = list[Callable[[Page], Awaitable[dict[str, Any]]]]


async def scrape(
    url: str,
    timeout: int = 20000,
    headless: bool | Literal["virtual"] = True,
    camoufox_options: dict[str, Any] | None = None,
) -> dict[str, Any]:

    captured_files = {"images": [], "videos": []}
    camoufox_options = camoufox_options or {}

    seen_urls: set[str] = set()

    parsed_url = urlparse(url)
    domain = parsed_url.netloc or parsed_url.path
    domain = re.sub(r"[^\w\-.]", "_", domain)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    session_dir = OUTPUT_DIR / f"{domain}_{timestamp}"

    (session_dir / "images").mkdir(parents=True, exist_ok=True)
    (session_dir / "videos").mkdir(parents=True, exist_ok=True)

    download_queue: asyncio.Queue = asyncio.Queue()

    workers = [
        asyncio.create_task(
            download_worker(download_queue, captured_files, session_dir)
        )
        for _ in range(NUM_WORKERS)
    ]

    text_data = {"name": "extract_text", "data": [], "count": 0}
    metadata = {"name": "extract_metadata", "data": {}}
    screenshot_path = session_dir / "screenshot.png"

    async with AsyncCamoufox(headless=headless, **camoufox_options) as browser:

        page = await browser.new_page()

        page.on(
            "response",
            lambda r: asyncio.create_task(
                on_response(r, download_queue, seen_urls, session_dir)
            ),
        )

        try:
            console.print(f"[cyan]Starting navigation (timeout={timeout}ms)...[/cyan]")
            await page.goto(url, wait_until="domcontentloaded", timeout=timeout)
            await page.wait_for_timeout(PAGE_LOAD_WAIT)

            console.print("[cyan]Scrolling to load lazy content...[/cyan]")
            scroll_info = await adaptive_scroll(page)
            console.print("[dim green]✓ Scroll complete[/dim green]")


            # non-blocking - continue even if it times out
            try:
                await page.wait_for_load_state("networkidle", timeout=2000)
                console.print("[dim green]✓ Network idle reached[/dim green]")
            except Exception:
                console.print(
                    "[dim yellow]⚠ Network still active, continuing anyway[/dim yellow]"
                )

            console.print("[cyan]Extracting page data...[/cyan]")
            screenshot_path = session_dir / "screenshot.png"

            text_data, metadata = await asyncio.gather(
                extract_text(page),
                extract_metadata(page),
            )

            screenshot_task = asyncio.create_task(
                page.screenshot(path=str(screenshot_path), full_page=False)
            )

            console.print("[dim green]✓ Page data extracted[/dim green]")

        except Exception as e:
            console.print(f"[yellow]Warning during scraping: {e}[/yellow]")
            console.print("[yellow]Continuing to save captured media...[/yellow]")
            text_data = {
                "name": "extract_text",
                "data": [],
                "count": 0,
                "error": str(e),
            }
            metadata = {"name": "extract_metadata", "data": {}, "error": str(e)}
            screenshot_task = None
        finally:
            console.print("[cyan]Waiting for all downloads to complete...[/cyan]")
            await download_queue.join()

            # Wait for screenshot if it's still running
            if 'screenshot_task' in locals() and screenshot_task:
                try:
                    await screenshot_task
                    console.print("[dim green]✓ Screenshot saved[/dim green]")
                except Exception as e:
                    console.print(f"[dim yellow]Screenshot failed: {e}[/dim yellow]")

        for _ in range(NUM_WORKERS):
            await download_queue.put(None)
        await asyncio.gather(*workers)

    total_count = len(captured_files["images"]) + len(captured_files["videos"])
    console.print(f"\n[bold green]✓ Download complete![/bold green]")
    console.print(f"[cyan]Total files:[/cyan] {total_count}")
    console.print(f"[cyan]Images:[/cyan] {len(captured_files['images'])}")
    console.print(f"[cyan]Videos:[/cyan] {len(captured_files['videos'])}")
    console.print(f"[cyan]Saved to:[/cyan] {session_dir}")

    result_data = {
        "url": url,
        "timestamp": timestamp,
        "session_dir": str(session_dir),
        "images": captured_files["images"],
        "videos": captured_files["videos"],
        "images_count": len(captured_files["images"]),
        "videos_count": len(captured_files["videos"]),
        "total_count": total_count,
        "text": text_data.get("data", []),
        "metadata": metadata.get("data", {}),
        "screenshot": str(screenshot_path) if screenshot_path.exists() else None,
    }

    result_json = session_dir / "scrape_result.json"
    async with aiofiles.open(result_json, "w") as f:
        await f.write(json.dumps(result_data, indent=2))

    console.print(f"[cyan]Results saved to:[/cyan] {result_json}")

    return result_data


In [ ]:
# Don't need these 2 lines when on local python
import nest_asyncio
nest_asyncio.apply()

# url = "https://magichour.ai"
# url = "https://www.amazon.ca/OCOOPA-Rechargeable-Magnetic-Handwarmers-Certified/dp/B0CC189314/"
url = "https://medium.com/@amit25173/scrapy-vs-beautifulsoup-vs-selenium-579bce149262"
# url = "https://www.cbc.ca/news/canada/british-columbia/islamophobia-in-b-c-1.6576808"

# use with asyncio.run() instead on local python
res = await scrape(url)

Starting navigation (timeout=20000ms)...

Scrolling to load lazy content...

Found 6 images and 0 videos on initial page load

Using 10.0% scroll step with 50ms delay

✓ Scroll complete

...

Total files: 7

Images: 7

Videos: 0

Saved to: scraped_data/medium.com_20251027_071742

Results saved to: scraped_data/medium.com_20251027_071742/scrape_result.json